In [35]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
import re
import string
import joblib

### Dataset

In [23]:
df = pd.read_csv("resume_job_matching_dataset.csv")
df.head()

,job_description,resume,match_score
0,"Data Analyst needed with experience in SQL, Ex...","Experienced professional skilled in SQL, Power...",4
1,Data Scientist needed with experience in Stati...,"Experienced professional skilled in Python, De...",4
2,Software Engineer needed with experience in Sy...,"Experienced professional skilled in wait, Git,...",5
3,"ML Engineer needed with experience in Python, ...","Experienced professional skilled in return, De...",4
4,Software Engineer needed with experience in RE...,"Experienced professional skilled in REST APIs,...",5


In [24]:
df.groupby("match_score").size()

match_score
1     440
2    1890
3    2269
4    3029
5    2372
dtype: int64

### Preprocessing

In [25]:
def clean_text(text):
  if pd.isnull(text):
      return ""
  text = text.lower()
  text = re.sub(f"[{string.punctuation}]", " ", text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()



df["job_description"] = df["job_description"].apply(clean_text)
df["resume"] = df["resume"].apply(clean_text)

### Feature extraction

In [26]:
combined_text = df["job_description"].tolist() + df["resume"].tolist()


# Tfidf
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf.fit(combined_text)

job_tfidf = tfidf.transform(df["job_description"])
resume_tfidf = tfidf.transform(df["resume"])


# Cosine similarity
cos_sim = []
for i in range(len(df)):
  sim = cosine_similarity(job_tfidf[i], resume_tfidf)[0][0]
  cos_sim.append(sim)
  
df["cosine_similarity"] = cos_sim



# Jaccard similarity
def jaccard_similarity(a, b):
  a_set, b_set = set(a.split()), set(b.split())
  if len(a_set | b_set) == 0:
    return 0
  return len(a_set & b_set) / len(a_set | b_set)

jacc_sim = [jaccard_similarity(df['resume'][i], df['job_description'][i]) for i in range(len(df))]



# Length ratio
len_ratio = [len(df['resume'][i].split()) / (len(df['job_description'][i].split()) + 1) for i in range(len(df))]

### Model

In [27]:
X = pd.DataFrame({
  "cosine_similarity": cos_sim,
  "jaccard_similarity": jacc_sim,
  "length_ratio": len_ratio
})
y = df["match_score"] * 20


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=1, min_samples_split=5, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

### Model performance

In [28]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 score: {r2:.2f}")

Model Performance:
MAE: 11.56
RMSE: 14.60
R2 score: 0.61


### Hyperparameter tuning

In [ ]:
rf = RandomForestRegressor(random_state=42)

param_dist = {
    "n_estimators": np.arange(100, 500, 100),   
    "max_depth": [None, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]    
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=100,            # number of random combinations
    cv=5,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
best_model = random_search.best_estimator_


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'n_estimators': np.int64(400), 'min_samples_split': 15, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 15}


### Updated model

In [29]:
model_rf = RandomForestRegressor(n_estimators=400, min_samples_split=15, min_samples_leaf=2, max_features='log2', max_depth=15)

model_rf.fit(X_train, y_train)

y_pred2 = model_rf.predict(X_test)



# Perfomance metrics
mae = mean_absolute_error(y_test, y_pred2)
rmse = np.sqrt(mean_squared_error(y_test, y_pred2))
r2 = r2_score(y_test, y_pred2)

print("Model Performance:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 score: {r2:.2f}")

Model Performance:
MAE: 11.64
RMSE: 14.53
R2 score: 0.61


In [37]:
joblib.dump(model_rf, "model.pkl")
joblib.dump(tfidf, "vectorizer.pkl")

['vectorizer.pkl']

### Prediction

In [30]:
def predict_match(job_desc, resume_text):
    # Clean inputs
    job_desc = clean_text(job_desc)
    resume_text = clean_text(resume_text)

    # TF-IDF vectors
    job_vec = tfidf.transform([job_desc])
    resume_vec = tfidf.transform([resume_text])

    # Feature 1: Cosine similarity
    cos_sim = cosine_similarity(job_vec, resume_vec)[0][0]

    # Feature 2: Jaccard similarity
    def jaccard_similarity(a, b):
        a_set, b_set = set(a.split()), set(b.split())
        if len(a_set | b_set) == 0:
            return 0
        return len(a_set & b_set) / len(a_set | b_set)

    jacc_sim = jaccard_similarity(resume_text, job_desc)

    # Feature 3: Length ratio
    len_ratio = len(resume_text.split()) / (len(job_desc.split()) + 1)

    # Put into same feature order as training
    features = [[cos_sim, jacc_sim, len_ratio]]

    # Predict and scale to 0–100
    pred = model_rf.predict(features)[0]
    return max(0, min(100, pred))


### Prediction example 1

In [31]:
job = """Web Development & Design Expert
$40-$100 / hr
Hourly contract
Remote
1. Role Overview
Mercor is seeking web development and design experts to support high-impact projects with leading AI labs and research organizations. We welcome specialists across front-end engineering, full-stack web, UX, UI, and product design. Freelancers will help build, evaluate, and refine web experiences—ensuring usability, performance, accessibility, and brand consistency.

2. Key Responsibilities
Opportunities could include tasks such as:
Designing information architecture, user flows, wireframes, and interactive prototypes
Building responsive, accessible web apps and sites (component libraries, design systems)
Implementing modern front-end stacks (e.g., React/Next.js, Vue/Nuxt, SvelteKit) and integrating APIs
Establishing visual systems (typography, color, spacing), and documenting design components
Conducting usability testing, heuristic reviews, and UX research synthesis
Optimizing for performance, SEO, analytics, and conversion (A/B tests, experiments)
Ensuring cross-browser/device compatibility and basic security and privacy hygiene
Reviewing and refining AI-generated UI, copy, and layouts; drafting prompts/playbooks for complex flows

3. Ideal Qualifications
2+ years in web development, product design, or UX/UI
Proficiency with modern front-end tools (JavaScript/TypeScript, React/Next.js or similar, CSS/Tailwind)
Experience with design tools (Figma, Sketch, or Adobe XD) and collaborative handoff
Strong grasp of accessibility (WCAG), responsive design, and component-driven development
Familiarity with testing and quality (Playwright/Cypress), analytics, and experimentation
Nice to have: headless CMS (Contentful, Sanity), Node.js, Webflow, motion/interaction design"""


resume = """Ujjwal Kumar
7061845104 | ujjwal.kumar.id@gmail.com | GitHub | LinkedIn 
EDUCATION
Maharaja Agrasen Institute of Technology - Delhi, India 2022 –Present
Bachelor of Technology in Electronics and Communication
CGPA: 7.8 / 10 (as of 6th semester)
PROJECTS
Authify | Next.js, TypeScript, MongoDB, JWT, Nodemailer
* Developed a full-stack authentication system with email verification and JWT-based login
* Integrated Nodemailer for email handling and Mongoose for database management
* Built responsive UI using Tailwind CSS and managed API calls with Axios
* GitHub
MailSense | Python, NLP, Flask, Tailwind CSS
* Developed a spam detection model using Multinomial Naive Bayes and NLP techniques
* Optimized with GridSearchCV and handled class imbalance using SMOTE
* Deployed via Flask web app for user interaction
* GitHub
Pokedox | Python, CNN, React.js, Tailwind CSS, FastAPI
* Built a CNN model to classify images into 10 Pokémon categories
* Trained on a custom image dataset and integrated with a full-stack web app
* Developed frontend in React.js with Tailwind CSS and backend in FastAPI
* GitHub
TECHNICAL SKILLS
Languages: C++, Python, JavaScript
Frontend: React.js, Tailwind CSS, HTML, CSS
Backend: Node.js, Express.js, RESTful APIs
Machine learning: NLP, Supervised & Unsupervised algorithms, Neural Networks
Misc: Flask, Streamlit, SQL, Git
CERTIFICATIONS
Complete Machine Learning Bootcamp – Udemy(Krish naik) July 2024
Web Development course – Udemy(Hitesh choudhary) July 2025"""

print(f"\n Match Percentage: {predict_match(job, resume):.2f}%")


 Match Percentage: 55.68%


c:\Users\saksh\OneDrive\Desktop\project_gp\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


### Prediction example 2

In [33]:
resume = "Experienced data analyst with Python, SQL, Power BI."
job = "Looking for a data analyst skilled in SQL, Excel, Power BI, ."
print(f"\n Match Percentage: {predict_match(job, resume):.2f}%")


 Match Percentage: 91.06%


c:\Users\saksh\OneDrive\Desktop\project_gp\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


### Prediction example 3

In [34]:
job = """We are seeking a skilled Nest.js Developer with 1- 2 years of experience to join our dynamic development team. The ideal candidate will have a solid foundation in backend development using Nest.js and a keen interest in building scalable and efficient web Responsibilities : 

 Develop and maintain backend applications using Nest.js.
 Design and implement RESTful APIs for seamless integration with frontend components.
 Collaborate with cross-functional teams to define, design, and ship new features.
 Ensure the performance, quality, and responsiveness of applications.
 Identify and correct bottlenecks and fix bugs.
 Help maintain code quality, organization, and Qualifications : 
 Bachelors degree in Computer Science, Information Technology, or a related field.
 1 - 2 years of professional experience in backend development using Nest.js.
 Proficiency in TypeScript and JavaScript.
 Experience with databases such as PostgreSQL, MySQL, or MongoDB.
 Familiarity with RESTful API design principles.
 Understanding of version control systems, preferably Git.
 Strong problem-solving skills and attention to Qualifications : 
 Experience with other backend frameworks like Express.js.
 Knowledge of frontend technologies such as React.js or Next.js.
 Familiarity with Docker and containerization concepts.
 Understanding of CI/CD pipelines and deployment processes.
"""

resume = """Ujjwal Kumar
7061845104 | ujjwal.kumar.id@gmail.com | GitHub | LinkedIn 
EDUCATION
Maharaja Agrasen Institute of Technology - Delhi, India 2022 –Present
Bachelor of Technology in Electronics and Communication
CGPA: 7.8 / 10 (as of 6th semester)
PROJECTS
Authify | Next.js, TypeScript, MongoDB, JWT, Nodemailer
* Developed a full-stack authentication system with email verification and JWT-based login
* Integrated Nodemailer for email handling and Mongoose for database management
* Built responsive UI using Tailwind CSS and managed API calls with Axios
* GitHub
MailSense | Python, NLP, Flask, Tailwind CSS
* Developed a spam detection model using Multinomial Naive Bayes and NLP techniques
* Optimized with GridSearchCV and handled class imbalance using SMOTE
* Deployed via Flask web app for user interaction
* GitHub
Pokedox | Python, CNN, React.js, Tailwind CSS, FastAPI
* Built a CNN model to classify images into 10 Pokémon categories
* Trained on a custom image dataset and integrated with a full-stack web app
* Developed frontend in React.js with Tailwind CSS and backend in FastAPI
* GitHub
TECHNICAL SKILLS
Languages: C++, Python, JavaScript
Frontend: React.js, Tailwind CSS, HTML, CSS
Backend: Node.js, Express.js, RESTful APIs
Machine learning: NLP, Supervised & Unsupervised algorithms, Neural Networks
Misc: Flask, Streamlit, SQL, Git
CERTIFICATIONS
Complete Machine Learning Bootcamp – Udemy(Krish naik) July 2024
Web Development course – Udemy(Hitesh choudhary) July 2025"""

print(f"\n Match Percentage: {predict_match(job, resume):.2f}%")


 Match Percentage: 78.05%


c:\Users\saksh\OneDrive\Desktop\project_gp\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
